In [ ]:
import pandas as pd
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data" / "raw"

x_train = pd.read_csv(DATA / "x_train.csv")
x_test = pd.read_csv(DATA / "x_test.csv")
y_train = pd.read_csv(DATA / "y_train.csv")



# Nettoyage des données

## nettoyage structurel

In [ ]:
print(x_train.info())


In [ ]:
print(y_train.info())


In [ ]:
print(x_train.shape)
print(x_train.head())


In [ ]:
print(x_train.isna().sum())


In [ ]:
print(x_train.nunique())





In [ ]:
print(x_train.duplicated().sum())

In [ ]:
print(y_train.shape)
print(y_train.head())


In [ ]:
print(y_train.isna().sum())


In [ ]:
print(y_train.duplicated().sum())

In [ ]:
print(y_train.nunique())

In [ ]:
X = x_train.drop(columns=["ID"])
X_test = x_test.drop(columns=["ID"])
Y = y_train.drop(columns=["ID","c15"])

## nettoyage statistique

In [ ]:
sns.set_theme(style="whitegrid")
print(X.shape, Y.shape)

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(x_train["Humidity"], label="x_train", fill=True, alpha=0.4)
sns.kdeplot(x_test["Humidity"], label="x_test", fill=True, alpha=0.4)
plt.title("Distribution de Humidity dans x_train et x_test")
plt.xlabel("Humidity")
plt.ylabel("Densité")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1 - Vue globale des distributions des features
X.hist(figsize=(16, 10), bins=30, edgecolor="black")
plt.suptitle("Distributions des variables de X", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 2 - Boxplots des features pour repérer les valeurs extrêmes

plt.figure(figsize=(14, 6))
sns.boxplot(data=X, orient="h")
plt.title("Boxplots des variables de X")
plt.show()

In [ ]:
# 4 - Densités des features
X.plot(kind="density", subplots=True, layout=(4, 4), figsize=(16, 12), sharex=False)
plt.suptitle("Densites des variables de X", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5 - Heatmap de corrélation des features
plt.figure(figsize=(10, 8))
corr_x = X.corr(numeric_only=True)
sns.heatmap(corr_x, cmap="coolwarm", center=0)
plt.title("Correlation entre les variables de X")
plt.show()

In [ ]:
# 6 - Distributions des targets
Y.hist(figsize=(18, 12), bins=30, edgecolor="black")
plt.suptitle("Distributions des variables de Y", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 7 - Boxplots des targets
plt.figure(figsize=(16, 8))
sns.boxplot(data=Y, orient="h")
plt.title("Boxplots des variables de Y")
plt.show()


In [ ]:
# 8 - Part de zeros par target
zero_share = (Y == 0).mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=zero_share.index, y=zero_share.values)
plt.title("Proportion de zeros par variable cible")
plt.ylabel("Proportion de zeros")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# 9 - Heatmap de corrélation des targets
plt.figure(figsize=(12, 10))
corr_y = Y.corr(numeric_only=True)
sns.heatmap(corr_y, cmap="coolwarm", center=0)
plt.title("Correlation entre les variables de Y")
plt.show()


In [ ]:
# 10 - Paires de variables X les plus corrélées
corr_pairs = (
    corr_x.where(np.triu(np.ones(corr_x.shape), k=1).astype(bool))
          .stack()
          .sort_values(key=np.abs, ascending=False)
)

corr_pairs.head(15)


### feature engineering

In [ ]:
X["M_high_mean"] = X[["M12","M13","M14","M15"]].mean(axis=1)
X["M_low_mean"]  = X[["M4","M5","M6","M7"]].mean(axis=1)

Ces groupes sont corrélés (heatmap), on les combine pour réduire le bruit individuel et pousser le modèle à lier ces données.

In [ ]:
X["M_high_std"] = X[["M12","M13","M14","M15"]].std(axis=1)
X["M_low_std"]  = X[["M4","M5","M6","M7"]].std(axis=1)

→ mesure la dispersion du signal
→ capteurs qui réagissent différemment
→ peut indiquer certains gaz

In [ ]:
X["M_high_max"] = X[["M12","M13","M14","M15"]].max(axis=1)
X["M_low_max"]  = X[["M4","M5","M6","M7"]].max(axis=1)

→ certains gaz activent fortement un capteur spécifique
→ max capture ce comportement

In [ ]:
X["M4_M5_ratio"] = X["M4"] / (X["M5"] + 1e-6)
X["M6_M7_ratio"] = X["M6"] / (X["M7"] + 1e-6)

→ capture relation relative
→ invariant à l’échelle
→ souvent très discriminant physiquement

In [ ]:
X["M_ratio"] = X["M_low_mean"] / (X["M_high_mean"] + 1e-6)

→ signature globale du gaz
→ interaction entre capteurs

In [ ]:
X["M4_H"] = X["M4"] * X["Humidity"]
X["M_mean_H"] = X["M_low_mean"] * X["Humidity"]

Intéraction avec humidity
→ capteurs sensibles à l’humidité
→ conditions physiques différentes
→ aide à gérer distribution shift

In [ ]:
import numpy as np

cols = ["M4","M5","M6","R","S2"]

for col in cols:
    X[f"log_{col}"] = np.log1p(np.abs(X[col]))

In [ ]:
→ réduit l’effet des grosses valeurs
→ stabilise les distributions
→ rend les patterns plus simples

In [ ]:
sensor_cols = ["M4","M5","M6","M7","M12","M13","M14","M15"]
X["sensor_var"] = X[sensor_cols].var(axis=1)

→ mesure la stabilité du signal
→ certains gaz → signal homogène
→ d’autres → très variable

In [ ]:
X["S_mean"] = X[["S1","S2","S3"]].mean(axis=1)

In [ ]:
X["R_S_ratio"] = X["R"] / (X["S_mean"] + 1e-6)

In [ ]:
→ ces capteurs sont corrélés
→ combinaison plus informative que séparément